# MiniMax H3 — FINAL ComfyUI + T4 Colab

This notebook is built around **ComfyUI**, not standalone DiffSynth. It uses the current ComfyUI/ClipProj H3 workflow, which provides text-to-video with native video+audio output. The ClipProj route replaces H3's 15.7 GB 32B text encoder with a Qwen3-VL 4B encoder plus a learned projection, which is much more appropriate for a 14.56 GB T4. The included workflow already contains `MiniMaxH3ImageToVideo`, audio VAE, video VAE, sampler, `CreateVideo`, and `SaveVideo`. citeturn0search0turn7view0turn9search2

**First generation defaults:** 384×224, 22 frames (~0.92 s at 24 FPS), 6 steps. Increase only after the first run works.


In [1]:
# CELL 1 — GPU CHECK. DO NOT SKIP.
import subprocess, torch
r=subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True,text=True)
print(r.stdout)
if r.returncode != 0 or not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. In Colab choose Runtime → Change runtime type → T4 GPU and reconnect.')
print('PyTorch:',torch.__version__)
print('CUDA:',torch.version.cuda)
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),'GB')
assert 'T4' in torch.cuda.get_device_name(0), 'This notebook is tuned for a T4.'


Sat Oct  3 17:43:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# CELL 2 — INSTALL COMFYUI + LOW-VRAM NODES
%cd /content
!rm -rf ComfyUI
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!python -m pip install -q -r requirements.txt
%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/nicolab28/ComfyUI-ClipProj.git
!git clone --depth 1 https://github.com/lericogit/ComfyUI-MiniMaxH3-LowVRAM.git
!git clone --depth 1 https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3.git
%cd /content/ComfyUI
!python - <<'PY'
import torch
print('Torch after ComfyUI install:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
PY


/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 1400, done.
remote: Counting objects: 100% (1400/1400), done.
remote: Compressing objects: 100% (1204/1204), done.
remote: Total 1400 (delta 152), reused 1222 (delta 150), pack-reused 0 (from 0)
Receiving objects: 100% (1400/1400), 12.24 MiB | 14.59 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/ComfyUI
/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI-ClipProj'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 26 (delta 1), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 803.47 KiB | 2.62 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into 'ComfyUI-MiniMaxH3-LowVRAM'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 13 (delta 0), reused 13 (delta 0), pack-reused 0 (fro

NameError: name 'PY' is not defined

### Why these nodes?

- **ClipProj** reduces the text-encoder memory requirement and supports dynamic/offloaded encoder loading; its current README recommends this approach for a single tight GPU. citeturn9search0turn9search2
- **MiniMax H3 Low VRAM** chunks H3 MLP activations. Its `low_vram` profile uses 4096-token chunks; it reduces MLP peak memory without intentionally approximating the model, although it cannot fix an attention OOM. citeturn10view0
- **Spectrum** is installed but not forced into the workflow because it is an approximate acceleration method and can alter the denoising trajectory. citeturn3search3


In [5]:
# CELL 3 — DOWNLOAD ONLY THE MODELS NEEDED BY THE 4B CLIPPROJ WORKFLOW
# The int8 ConvRot diffusion checkpoint is large (~34 GB).
# The 4B FP8 text encoder is ~5.2 GB. VAE files are ~5.2 GB + 0.6 GB.
!python -m pip install -q -U huggingface_hub
!mkdir -p /content/ComfyUI/models/text_encoders /content/ComfyUI/models/clip_projections /content/ComfyUI/models/diffusion_models /content/ComfyUI/models/vae
!hf download Comfy-Org/Krea-2 text_encoders/qwen3vl_4b_fp8_scaled.safetensors --local-dir /content/ComfyUI/models/text_encoders
!hf download NicoLab28/ClipProj-MiniMax-H3 mmh3-4b-ClipProj-v3.1-mlp.safetensors --local-dir /content/ComfyUI/models/clip_projections
!hf download Comfy-Org/MiniMax-H3 diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors --local-dir /content/ComfyUI/models/diffusion_models
!hf download Comfy-Org/MiniMax-H3 vae/minimax_h3_video_vae_fp16.safetensors --local-dir /content/ComfyUI/models/vae
!hf download Comfy-Org/MiniMax-H3 vae/minimax_h3_audio_vae_fp32.safetensors --local-dir /content/ComfyUI/models/vae
!df -h /content


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.7.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.17.0 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
Hint: The `hf-cli` skill is not installed. Run `hf skills a

In [ ]:
import os
import shutil

base = "/content/ComfyUI/models"

moves = [
    (
        f"{base}/vae/vae/minimax_h3_audio_vae_fp32.safetensors",
        f"{base}/vae/minimax_h3_audio_vae_fp32.safetensors"
    ),
    (
        f"{base}/vae/vae/minimax_h3_video_vae_fp16.safetensors",
        f"{base}/vae/minimax_h3_video_vae_fp16.safetensors"
    ),
    (
        f"{base}/diffusion_models/diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors",
        f"{base}/diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors"
    ),
]

for src, dst in moves:
    if os.path.exists(src):
        print("Moving:", src)
        shutil.move(src, dst)
        print("  ->", dst)
    elif os.path.exists(dst):
        print("Already correct:", dst)
    else:
        print("NOT FOUND:", src)

print("\n=== FINAL MODEL CHECK ===")

required = [
    f"{base}/vae/minimax_h3_audio_vae_fp32.safetensors",
    f"{base}/vae/minimax_h3_video_vae_fp16.safetensors",
    f"{base}/diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors",
    f"{base}/text_encoders/qwen3vl_4b_fp8_scaled.safetensors",
]

for f in required:
    if os.path.exists(f):
        print("✅", f, f"({os.path.getsize(f)/(1024**3):.2f} GB)")
    else:
        print("❌", f)

print("\n=== MODEL TREE ===")
os.system("find /content/ComfyUI/models -type f -printf '%p\\n' | sort")

**Storage note:** the current Comfy-Org int8 ConvRot FL2VA checkpoint is about **34 GB**, while the 4B FP8 encoder is about **5.2 GB**. These are the current files referenced by the ClipProj workflow. citeturn2search2turn7view0


In [6]:
# CELL 4 — DOWNLOAD + PATCH THE REAL COMFYUI WORKFLOW
import json, urllib.request, pathlib
COMFY='/content/ComfyUI'
workflow_url='https://raw.githubusercontent.com/nicolab28/ComfyUI-ClipProj/main/example_workflows/minimax_h3_clipproj.json'
raw=urllib.request.urlopen(workflow_url).read().decode('utf-8')
wf=json.loads(raw)


# User generation settings — change these after the first successful run.
PROMPT='''Photorealistic cinematic live-action video of a young Indian man in his early twenties standing on a modern city street at night immediately after rainfall. Realistic masculine face, natural skin texture, short black hair, expressive eyes, realistic anatomy and subtle facial movements. He wears a dark modern casual jacket. Wet pavement reflects colorful city lights and soft neon signs. Light rain continues falling around him. The camera slowly tracks forward toward him with natural handheld cinematic movement and realistic depth of field. Subtle wind moves his hair and jacket naturally. He looks directly into the camera and naturally says: "I love creating videos with MiniMax H3." Accurate natural lip synchronization, realistic mouth movement, natural blinking and facial expressions. Realistic environmental reflections, physically plausible lighting, cinematic color grading, detailed skin and fabric textures, realistic rain particles, natural motion. No subtitles, captions, text overlays, watermark, logo, cartoon, anime, illustration, CGI appearance, distorted face, extra fingers, deformed anatomy.'''

WIDTH=384
HEIGHT=224
FRAMES=22
STEPS=6
SEED=42


for n in wf['nodes']:
    if n['id']==1:  # ClipProjLoader
        n['widgets_values']=['qwen3vl_4b_fp8_scaled.safetensors','minimax','mmh3-4b-ClipProj-v3.1-mlp.safetensors','cuda:0','dynamic']
    elif n['id']==2:  # diffusion model
        n['widgets_values']=['minimax_h3_fl2va_int8_convrot.safetensors','default']
    elif n['id']==3:  # H3 sigma shift
        n['widgets_values']=[12.0,3.0]
    elif n['id']==4:
        n['widgets_values']=['minimax_h3_video_vae_fp16.safetensors']
    elif n['id']==5:
        n['widgets_values']=['minimax_h3_audio_vae_fp32.safetensors']
    elif n['id']==6:  # prompt + dimensions + duration
        n['widgets_values'][0]=PROMPT
        n['widgets_values'][1]=WIDTH
        n['widgets_values'][2]=HEIGHT
        n['widgets_values'][3]=FRAMES
    elif n['id']==9:  # scheduler steps
        n['widgets_values'][1]=STEPS
    elif n['id']==10:
        n['widgets_values'][0]=SEED
    elif n['id']==15:
        n['widgets_values'][0]='video/MiniMax_H3_T4'


# Insert the current Low-VRAM MLP patch between the diffusion loader and H3 sigma shift.
new_id=max(n['id'] for n in wf['nodes'])+1
new_link=max(x[0] for x in wf['links'])+1
low_node={
  'id':new_id,'type':'MiniMaxH3LowVRAM','pos':[40,350],'size':[360,240],'flags':{},'order':7,'mode':0,
  'inputs':[
    {'name':'model','type':'MODEL','link':new_link},
    {'name':'enabled','type':'BOOLEAN','link':None,'widget':{'name':'enabled'}},
    {'name':'memory_profile','type':'COMBO','link':None,'widget':{'name':'memory_profile'}},
    {'name':'custom_chunk_tokens','type':'INT','link':None,'widget':{'name':'custom_chunk_tokens'}},
    {'name':'block_prefetch','type':'COMBO','link':None,'widget':{'name':'block_prefetch'}},
    {'name':'verbose','type':'BOOLEAN','link':None,'widget':{'name':'verbose'}}],
  'outputs':[{'name':'MODEL','type':'MODEL','links':[]}],
  'properties':{'Node name for S&R':'MiniMaxH3LowVRAM'},
  'widgets_values':[True,'low_vram',4096,'disable',False],
  'title':'T4 Low-VRAM MLP Patch'
}
wf['nodes'].append(low_node)


# Existing loader→sigma link becomes loader→LowVRAM.
for link in wf['links']:
    if link[0]==2 and link[3]==3:
        link[3]=new_id
        link[4]=0
        link[5]='MODEL'
# Add LowVRAM→sigma. SigmaShift output links remain unchanged.
wf['links'].append([new_link,new_id,0,3,0,'MODEL'])
# Update sigma node input link to the new link.
for n in wf['nodes']:
    if n['id']==3:
        for inp in n['inputs']:
            if inp['name']=='model': inp['link']=new_link

out='/content/ComfyUI/workflow_minimax_h3_t4.json'
with open(out,'w',encoding='utf-8') as f: json.dump(wf,f,indent=2)
print('Patched workflow:',out)
print(f'Settings: {WIDTH}x{HEIGHT}, {FRAMES} frames, {STEPS} steps, seed {SEED}')


Patched workflow: /content/ComfyUI/workflow_minimax_h3_t4.json
Settings: 384x224, 22 frames, 6 steps, seed 42


### Frame rule
H3 uses the `17n+5` frame grid at 24 FPS. 22 frames is ~0.92 seconds; 39 is ~1.63 s; 56 is ~2.33 s; 73 is ~3.04 s; 90 is ~3.75 s; 107 is ~4.46 s; 124 is ~5.17 s. citeturn0search1turn7view0


In [28]:
# CELL 5 — START COMFYUI FOR PUBLIC ACCESS

%cd /content/ComfyUI

import subprocess
import time
import sys

cmd = [
    sys.executable,
    "main.py",

    "--listen", "0.0.0.0",
    "--port", "8188",

    "--lowvram",
    "--fp16-vae",

    # Allow requests coming through the public tunnel
    "--enable-cors-header", "*"
]

log = open("/content/comfyui.log", "w")

proc = subprocess.Popen(
    cmd,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(12)

print("ComfyUI PID:", proc.pid)
print("\n===== COMFYUI LOG =====\n")
print(open("/content/comfyui.log").read()[-5000:])

/content/ComfyUI
ComfyUI PID: 12004

===== COMFYUI LOG =====

uantize_int8_simple_dtype', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'dequantize_w4a8_int8_weight', 'fp16_conv3d', 'fp16_conv3d_out', 'fp16_linear', 'gemv_awq_w4a16', 'group_norm_silu_pad3d', 'group_norm_silu_pad3d_out', 'int8_linear', 'na3d', 'prepare_int4_weight_for_int8_linear', 'quantize_and_rotate_rowwise', 'quantize_convrot_w4a4_weight', 'quantize_int8_convrot_weight', 'quantize_int8_rowwise', 'quantize_int8_tensorwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4', 'quantize_w4a8_int8_weight', 'rms_adaln', 'rms_rope', 'rms_rope1', 'rms_rope1_', 'rms_rope_', 'rms_rope_split_half', 'rms_rope_split_half1', 'rms_rope_split_half1_', 'rms_rope_split_half_', 'rotate_int8_convrot_weight', 'scaled_mm_mxfp8', 'scaled_mm_nvfp4', 'scaled_mm_svdquant_w4a4', 'sol_attn', 'stochastic_rounding_fp8', 'w4a8_int8_linear']}
[INFO] Found comfy_kitchen backend hip: {'ava

In [29]:
# VERIFY COMFYUI

import requests

try:
    r = requests.get(
        "http://127.0.0.1:8188",
        timeout=10
    )

    print("Status:", r.status_code)

    if r.status_code == 200:
        print("🔥 COMFYUI LOCAL ACCESS WORKS")
    else:
        print("⚠️ Unexpected status")

except Exception as e:
    print("❌", e)

Status: 200
🔥 COMFYUI LOCAL ACCESS WORKS


In [16]:
# NEW CELL — VERIFY COMFYUI

import requests
import time

time.sleep(3)

try:
    r = requests.get("http://127.0.0.1:8188", timeout=10)

    if r.status_code == 200:
        print("🔥🔥🔥 COMFYUI IS RUNNING!")
        print("Local: http://127.0.0.1:8188")
    else:
        print("⚠️ ComfyUI responded with:", r.status_code)

except Exception as e:
    print("❌ ComfyUI still isn't running.")
    print(e)

    print("\n===== LAST LOG =====")
    print(open("/content/comfyui.log").read()[-6000:])

🔥🔥🔥 COMFYUI IS RUNNING!
Local: http://127.0.0.1:8188


In [30]:
# CELL 6 — CLOUDFLARE TUNNEL

import os
import subprocess
import time
import re

os.system("pkill -f cloudflared 2>/dev/null || true")
time.sleep(2)

if not os.path.exists("/tmp/cloudflared"):
    os.system(
        "wget -q "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /tmp/cloudflared"
    )

os.system("chmod +x /tmp/cloudflared")

tunnel = subprocess.Popen(
    [
        "/tmp/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8188",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None

start = time.time()

while time.time() - start < 60:

    line = tunnel.stdout.readline()

    if line:
        print(line.rstrip())

        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line
        )

        if match:
            public_url = match.group(0)
            break

print("\n========================================")

if public_url:
    print("🔥 NEW COMFYUI URL")
    print(public_url)
else:
    print("❌ URL not found")

print("========================================")

2026-10-03T18:25:42Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-03T18:25:42Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-03T18:25:46Z INF +--------------------------------------------------------------------------------------------+
2026-10-03T18:25:46Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-03T18:25:46Z INF |  https://began-attached-california-heard.trycloudflare

In [36]:
# CHECK COMFYUI BACKEND + WEBSOCKET ENDPOINT

import requests

print("=== ROOT ===")
r = requests.get("http://127.0.0.1:8188/")
print(r.status_code)

print("\n=== SYSTEM STATS ===")
r = requests.get("http://127.0.0.1:8188/system_stats")
print(r.status_code)
print(r.text[:500])

print("\n=== QUEUE ===")
r = requests.get("http://127.0.0.1:8188/queue")
print(r.status_code)
print(r.text[:500])

=== ROOT ===
200

=== SYSTEM STATS ===
200
{"system": {"os": "linux", "ram_total": 13605830656, "ram_free": 11051532288, "comfyui_version": "0.38.0", "required_frontend_version": "1.53.10", "installed_templates_version": "0.11.76", "required_templates_version": "0.11.76", "comfy_package_versions": [{"name": "comfyui-frontend-package", "installed": "1.53.10", "required": "1.53.10"}, {"name": "comfyui-workflow-templates", "installed": "0.11.76", "required": "0.11.76"}, {"name": "comfyui-embedded-docs", "installed": "0.5.13", "required": "0

=== QUEUE ===
200
{"queue_running": [], "queue_pending": []}


### In the ComfyUI window
1. Load `/content/ComfyUI/workflow_minimax_h3_t4.json` by dragging it into the ComfyUI canvas.
2. The workflow is already patched for the 4B ClipProj encoder, current H3 int8 ConvRot diffusion model, H3 video/audio VAEs, T4 low-VRAM MLP patch, your cinematic prompt, and a 22-frame test.
3. Click **Queue Prompt**.
4. The resulting MP4 is written under `/content/ComfyUI/output/video/`.

ComfyUI's native H3 documentation confirms that H3 can generate native stereo audio and that the T2V workflow uses the H3 video/audio VAEs and a 17k+5 duration grid. citeturn0search0turn0search1


In [42]:
# CELL 7 — FIND THE GENERATED MP4
!find /content/ComfyUI/output -type f \( -iname '*.mp4' -o -iname '*.webm' \) -printf '%T@ %p\n' 2>/dev/null | sort -nr | head -10


1791053605.2509452220 /content/ComfyUI/output/video/MiniMax_H3_T4_00001_.mp4


In [43]:
# CELL 8 — OPTIONAL: AFTER A SUCCESSFUL 22-FRAME RUN, TRY ~3 SECONDS
# Edit the values in CELL 4 and rerun CELL 4, then restart/refresh ComfyUI before queuing.
print('Recommended next test: WIDTH=384, HEIGHT=224, FRAMES=73, STEPS=6')
print('Do not jump directly to 832x480/124 frames on a T4.')


Recommended next test: WIDTH=384, HEIGHT=224, FRAMES=73, STEPS=6
Do not jump directly to 832x480/124 frames on a T4.
